# 07 — 连续时序隐变量：卡尔曼滤波与观测 MLE
沿用第 06 章的时间递推与序列 MLE，同时采用第 03 章的连续线性高斯观测模型。第 06 章的离散求和现在变成连续积分；线性高斯结构使预测与更新可以解析计算，不需要对整个状态轨迹做高维网格积分。仍是二维观测、标量隐变量。模型假设改变后，重新生成符合时序结构的数据。

$x_t=ax_{t-1}+w_t,\; w_t\sim N(0,Q)$；$y_t=h x_t+b+v_t,\; v_t\sim N(0,R)$。其中 $h$ 为二维向量，$R$ 为二维协方差矩阵，噪声独立，初始状态 $x_0\sim N(0,1)$。仅 $a$ 未知，其余参数固定。

目标：在不读取真实状态的情况下估计 $a$。滤波递推时，观测预测分布为 $N(hm_t^-+b,\; hP_t^-h^T+R)$。

给定上一时刻后验均值 m、方差 P，预测先验均值 $m^-=am$、方差 $P^-=a^2P+Q$。边缘观测分布的均值 $\bar y=hm^-+b$、协方差 $S=P^-hh^T+R$。贝叶斯更新的解析结果为 $K=P^-h^TS^{-1}$（代码使用一维数组表示行向量），$m^+=m^-+K(y-\bar y)$、$P^+=P^--KhP^-$。这里直接使用线性高斯解析积分与解析后验，不需要枚举状态。

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
from utils import plot_likelihood
rng=np.random.default_rng(42)
T = 500
a_true = .85
Q = .2
R = .45**2*np.eye(2)
h = np.array([1.,.8])
b = np.array([0.,1.])
# 生成仿真真值；这些状态不传入滤波器。
x = np.empty(T)
x[0] = rng.normal()
for t in range(1,T):
    x[t]=a_true*x[t-1]+np.sqrt(Q)*rng.normal()
noise = rng.multivariate_normal(np.zeros(2),R,size=T)
y = np.zeros((T,2))
for t in range(T):
    y[t] = h*x[t]+b+noise[t]
def kalman(y,a):
    # 初始状态先验 N(0,1)，m 为均值，P 为方差。
    m, P = 0., 1.
    estimates = []
    variances = []
    ll = []
    for t,obs in enumerate(y):
        # 预测先验 p(x_t | 历史观测) 的均值与方差。
        if t == 0:
            prior_mean, prior_variance = m, P
        else:
            prior_mean = a * m
            prior_variance = a**2 * P + Q

        # 对状态边缘化：解析观测分布 p(y_t | 历史观测)。
        observation_mean = h * prior_mean + b
        observation_covariance = prior_variance * np.outer(h, h) + R
        marginal = multivariate_normal(observation_mean, observation_covariance)
        log_marginal = marginal.logpdf(obs)
        ll.append(log_marginal)

        # 贝叶斯后验的解析均值和方差。
        # S 对称，solve(S, P*h) 等价于求 K 的转置，
        # 不显式求逆矩阵，避免额外数值误差。
        K = np.linalg.solve(observation_covariance, prior_variance * h)
        innovation = obs - observation_mean
        posterior_mean = prior_mean + K @ innovation
        posterior_variance = prior_variance - (K @ h) * prior_variance
        m, P = posterior_mean, posterior_variance
        estimates.append(m)
        variances.append(P)
    return np.array(estimates),np.array(variances),np.sum(ll)
def negative_log_likelihood(a):
    _, _, ll = kalman(y,a)
    return -ll
# 与第 06 章相同的序列 MLE；这里使用一维有界优化器。
result=minimize_scalar(negative_log_likelihood,bounds=(-.99,.99),method='bounded')
a_hat=result.x
m,P,ll=kalman(y,a_hat)
print('MLE a:',a_hat,'simulation truth:',a_true)
t=np.arange(100)
plt.plot(t,x[:100],label='truth')
plt.plot(t,m[:100],label='filter')
plt.fill_between(t,m[:100]-2*np.sqrt(P[:100]),m[:100]+2*np.sqrt(P[:100]),alpha=.2)
plt.legend()
plt.show()

真值 x 仅用于生成数据与绘图。参数学习目标只依赖观测 y。与第 06 章一样，滤波提供序列似然，外层优化器负责学习参数 a。这里用解析高斯递推替代离散求和。下一段沿用第 06 章的网格扫描，展示同一个似然目标。

In [ ]:
candidates = np.linspace(-.95,.98,50)
likelihood = []
for a in candidates:
    _, _, ll = kalman(y,a)
    likelihood.append(ll)
plot_likelihood(candidates, likelihood, a_hat, 'MLE')

如果将线性状态转移或线性观测映射换成非线性神经网络，高斯噪声仍可保留，但预测状态和后验一般不再是高斯。这时需要近似推断，或在低维情况下进行数值积分。概率世界模型可以从这里进一步扩展；本项目未实现完整 PlaNet。